### Deduplication methods
pyspark code to remove duplicates from a table

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col

# Initialize Spark session
spark = SparkSession.builder.appName("DeduplicationExample").getOrCreate()

# Define schema
schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("timestamp", StringType(), True)
])

# Create sample data with multiple deduplication cases
data = [
    # Case 1: Exact duplicates (should be deduped)
    ("1001", "Alice", 250, "2025-06-20 10:00:00"),
    ("1001", "Alice", 250, "2025-06-20 10:00:00"),
    
    # Case 2: Null order IDs (should be rejected)
    (None, "Charlie", 300, "2025-06-20 10:10:00"),
    (None, "Hank", 500, "2025-06-20 10:35:00"),
    
    # Case 3: Same order ID with different timestamps (should dedupe keeping first)
    ("1002", "Bob", 150, "2025-06-20 10:05:00"),
    ("1002", "Bob", 150, "2025-06-20 11:05:00"),
    
    # Case 4: Same order ID with different amounts (should dedupe keeping first)
    ("1003", "David", 200, "2025-06-20 10:15:00"),
    ("1003", "David", 250, "2025-06-20 10:15:00"),
    
    # Case 5: Unique records (should be kept)
    ("1004", "Eve", 350, "2025-06-20 10:20:00"),
    ("1005", "Frank", 400, "2025-06-20 10:25:00"),
    
    # Case 6: Duplicate with different customer name (should dedupe keeping first)
    ("1006", "Grace", 450, "2025-06-20 10:30:00"),
    ("1006", "Grace-Updated", 450, "2025-06-20 10:35:00")
]

# Create DataFrame
df = spark.createDataFrame(data, schema)


In [0]:
df.show()
print("no of records : ",df.count())

In [0]:
# removing null values using dropna()
df_clean = df.dropna()
df_clean.show()
print("no of records after dropping nulls :",df_clean.count())

In [0]:
# dedup table based on all the columns
df_distinct = df_clean.distinct()
df_distinct.show()
print("no of records after distinct operation :", df_distinct.count())

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, rank, row_number

w = Window.partitionBy("order_id","amount","timestamp").orderBy("timestamp")
df_ranked = df_clean.withColumn("rn",row_number().over(w))
df_ranked.show()

In [0]:
df_deduped = df_ranked.filter(col("rn") == 1)
df_deduped = df_deduped.drop("dr")
df_deduped.show()
print("no of records after dededupe :", df_deduped.count())

### DataFrames

### Merging dataframes

In [0]:
# Merge two dataframes
data1 = [
    (1,"Rohan Srivastwa","AI_ML","Patna",9),
    (2,"Shubham","MBA","Mumbai",5 ),
    (3,"Samyak Jain","NABARD","Gangtok",7)]
columns1 = ["ID", "Student_Name", "Department_Name", "City", "Marks"]
df_1 = spark.createDataFrame(data = data1, schema = columns1)
df_1.show()

In [0]:
data2 = [(5,"Sudhir","Delhi",1),
         (16,"Amisha","Bnaras",3)]
columns2 = ["id", "StudentName", "City","Score"]
df_2 = spark.createDataFrame(data=data2, schema=columns2)
df_2.show()

In [0]:
df_1.union(df_2)

In [0]:
# Solution :
from pyspark.sql.functions import lit
df_2 = df_2.withColumn("Department_Name", lit("null"))
df_2.show()

In [0]:
df_1.columns

In [0]:
df_2.columns

In [0]:
# to alter the order of columns in dataframe
from pyspark.sql.functions import col
df_ordered = df_2.select(col("id"),col("StudentName"),col("Department_Name"), col("City"), col("Score").alias("Marks"))
df_ordered.show()

In [0]:
df = df_1.union(df_ordered)
df.show()

In [0]:
# Explode columns 
data = [
    (1, ["Python", "SQL", "Spark"]),
    (2, ["Java", "Scala"]),
    (3, [])
]
df = spark.createDataFrame(data, ["id", "skills"])
df.show(truncate=False)

In [0]:
from pyspark.sql.functions import explode, explode_outer
df.select("id", explode("skills")).show()

In [0]:
from pyspark.sql.functions import explode
df.select("id", explode_outer("skills")).show()

### Regular expressions

In [0]:
dbutils.fs.rm('dbfs:/user/hive/warehouse/student',True)

In [0]:
spark.sql("DROP TABLE IF EXISTS student;")
spark.sql("""
CREATE TABLE student (
    id INT,
    student_name STRING,
    mobile_no STRING);
    """)
spark.sql("""
INSERT INTO student (id, student_name,mobile_no) VALUES
(2, 'Shivam', 'U6753679iy'),
(1, 'Sagar', '7654367862'),
(3, 'Muni', '89765415U7');
        """)

In [0]:
dbutils.fs.ls('dbfs:/user/hive/warehouse/student')

In [0]:
%sql
SELECT * FROM student;

In [0]:
student_df = spark.sql("SELECT * FROM student")
student_df.show()

In [0]:
from pyspark.sql.functions import col, trim
student_df.select("*").filter(trim(col("mobile_no")).rlike('^[0=9]+$')).show()

In [0]:
%sql
SELECT * FROM student WHERE Mobile_No LIKE "^[0-9]*$"

In [0]:
# to filter columns having only alphanumeric values
spark.sql("SELECT * FROM student WHERE mobile_no LIKE '^[a-zA-Z0-9]+$ '").show()

In [0]:
spark.sql("SELECT * FROM student").show()

In [0]:
spark.sql("SELECT * FROM student WHERE mobile_no LIKE 'U%'").show()

### Loading files from various sources 
skip lines while loading data into dataframes

#### Excel

#### CSV

In [0]:
df = spark.read.format("csv").load("")

Accenture interview - https://www.youtube.com/watch?v=iZzP3CMq8xE&ab_channel=GeekCoders
Write a PySpark query to report the movies with odd-numbered ID and a description that is not "boring". Return the result table in descending order by rating.

In [0]:
data=[(1, 'War','great 3D',8.9)
,(2, 'Science','fiction',8.5) 
,(3, 'irish','boring',6.2)
,(4, 'Ice song','Fantacy',8.6)  
,(5, 'House card','Interesting',9.1)]    
schema="ID int,movie string,description string,rating double"
df=spark.createDataFrame(data,schema)
df.show()

In [0]:
from pyspark.sql.functions import col
df.filter((col('description') != 'boring') & (col('ID')%2 == 1)).orderBy(col("rating").desc()).show()

Solution 2 - Using SQL

In [0]:
df.createTempView("movie")

In [0]:
spark.sql('''SELECT * FROM movie WHERE description != "boring" AND ID%2 =1 ORDER BY rating DESC
          ''').show()

<pre>
<b> input data </b>
john	tomato	2
𝚋𝚒𝚕𝚕	𝚊𝚙𝚙𝚕𝚎	2
john	𝚋𝚊𝚗𝚊𝚗𝚊	2
john	tomato	3
𝚋𝚒𝚕𝚕	𝚝𝚊𝚌𝚘	2
𝚋𝚒𝚕𝚕	𝚊𝚙𝚙𝚕𝚎	2


<b> output data </b>
john	tomato	5
𝚋𝚒𝚕𝚕	𝚊𝚙𝚙𝚕𝚎	4
john	𝚋𝚊𝚗𝚊𝚗𝚊	2
𝚋𝚒𝚕𝚕	𝚝𝚊𝚌𝚘	2

</pre>


In [0]:
data = [
    ("john", "tomato", 2),
    ("𝚋𝚒𝚕𝚕", "𝚊𝚙𝚙𝚕𝚎", 2),
    ("john", "𝚋𝚊𝚗𝚊𝚗𝚊", 2),
    ("john", "tomato", 3),
    ("𝚋𝚒𝚕𝚕", "𝚝𝚊𝚌𝚘", 2),
    ("𝚋𝚒𝚕𝚕", "𝚊𝚙𝚙𝚕𝚎", 2),
]
schema = "name string, item string, weight int"
df = spark.createDataFrame(data, schema)
df.show(truncate=True)
df.printSchema()

In [0]:
display(df)

In [0]:
df_agg = df.groupBy("name", "item").sum("weight").alias("sum_weight")
display(df_agg)

In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
df = df.withColumn("weight", col("weight").cast("integer"))

In [0]:
df.printSchema()

In [0]:
df.columns

In [0]:
df.select("weight").distinct().show()

In [0]:
from pyspark.sql.functions import sum, col, collect_list, struct, collect_set
df_agg_alias = df.groupBy("name", "item").agg(sum("weight").alias("sum_weight"))
display(df_agg_alias)


In [0]:
df_collect_list_struct = df_agg_alias.groupBy("name").agg(collect_list(struct('item', 'sum_weight')).alias("items")).orderBy(col("name").desc())
display(df_collect_list_struct)

In [0]:
df_collect_list_struct = df_agg_alias.groupBy("name").agg(collect_set(struct('item', 'sum_weight')).alias("items")).orderBy(col("name").desc())
display(df_collect_list_struct)

#### SFTP

#### Databases

##### SQL Server

In [0]:
# read from sql server - JDBC connection
df.read.format("jdbc")

# to write to SQL Server - JDBC connection
df.write.format("jdbc")\
        .option("url","jdbc:sqlserver://<host>")\
        .option(f"{db_name}",f"{table_name}")\
        .option("batchsize", 100000)   
        .save()

##### Oracle 

In [0]:
print("a")